# Multi-task ResNet18 Attribute Training and Test

Notebook train chung shape/color cho head-tune va last-block fine-tune, dong thoi ho tro test checkpoint co san. Chon `EXECUTION_MODE` o Cell cau hinh. Bat Internet trong Kaggle de clone repo, va attach thu muc `data/` co `image_all/nih_attribute/`, `splits/`, `processed/`; che do test-only can attach them dataset model artifact.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = 'https://github.com/GOx9-P/Multiple-Pill-Recognition-And-Interaction-Safety.git'
BRANCH = 'CV_attribute_ResNet18_NguyenGiaBao'
REPO_DIR = Path('/kaggle/working/Multiple-Pill-Recognition-And-Interaction-Safety')

if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)

print('Repository:', REPO_DIR)
print('Branch:', BRANCH)

In [ ]:
# Dung cac goi NumPy/SciPy/Pillow co san cua image Kaggle.
# Khong force-reinstall cac goi nen: doi NumPy khi SciPy da duoc kernel nap se gay loi ABI o Cell 5.
numeric_check = subprocess.run(
    [sys.executable, '-c', 'import numpy; import scipy; import sklearn'],
    text=True,
    capture_output=True,
)
if numeric_check.returncode != 0:
    # Chi dung de sua session da bi loi boi cac lan cai dat cu; session moi se khong vao nhanh nay.
    print('Detected broken NumPy/SciPy environment. Repairing compatible numeric packages...', flush=True)
    subprocess.run([
        sys.executable, '-m', 'pip', 'uninstall', '-y',
        'numpy', 'scipy', 'scikit-learn'
    ], check=True)
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir',
        'numpy==1.26.4', 'scipy==1.13.1', 'scikit-learn==1.5.2'
    ], check=True)
    print('Numeric packages repaired. Restarting the notebook kernel now...', flush=True)
    import os
    os._exit(0)

# Process con co the doc package moi trong khi kernel hien tai van giu NumPy cu trong RAM.
# Kiem tra lai ngay trong kernel de khong day loi ABI xuong Cell import workflow.
try:
    import numpy as np
    import scipy
    import sklearn
    from scipy.sparse import csr_matrix
except Exception as numeric_kernel_error:
    print(
        'Numeric packages on disk are healthy but this kernel is stale: '
        f'{type(numeric_kernel_error).__name__}: {numeric_kernel_error}',
        flush=True,
    )
    print('Restarting the notebook kernel before workflow imports...', flush=True)
    import os
    os._exit(0)

import PIL
from PIL import Image, ImageDraw
print('Pillow:', PIL.__version__)

# Probe trong process rieng de chua import torch vao kernel notebook.
probe_code = (
    "import torch; "
    "cap=torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (-1,-1); "
    "print('TORCH_PROBE|' + torch.__version__ + '|' + str(torch.version.cuda) + '|' + "
    "str(cap[0]) + '.' + str(cap[1]) + '|' + ','.join(torch.cuda.get_arch_list()))"
)
probe = subprocess.run([sys.executable, '-c', probe_code], text=True, capture_output=True)
probe_line = next(
    (line for line in probe.stdout.splitlines() if line.startswith('TORCH_PROBE|')),
    None,
)

needs_compatible_wheel = probe.returncode != 0 or probe_line is None
if probe_line is not None:
    _, old_torch, old_cuda, capability, arch_text = probe_line.split('|', 4)
    required_arch = 'sm_' + capability.replace('.', '')
    needs_compatible_wheel = required_arch not in arch_text.split(',')
    print('Existing torch:', old_torch, '| CUDA:', old_cuda, '| capability:', capability)
    print('Existing CUDA architectures:', arch_text)

if needs_compatible_wheel:
    print('Current PyTorch wheel does not support this GPU. Installing compatible cu124 wheel...')
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--force-reinstall',
        'torch==2.6.0', 'torchvision==0.21.0', 'torchaudio==2.6.0',
        '--index-url', 'https://download.pytorch.org/whl/cu124'
    ], check=True)
    if 'torch' in sys.modules:
        raise RuntimeError(
            'Da cai PyTorch cu124. Hay Restart Session, sau do Run All de nap wheel moi.'
        )

import torch
import torchvision
if not torch.cuda.is_available():
    raise RuntimeError('Hay bat GPU accelerator trong Kaggle Settings truoc khi train.')

required_arch = 'sm_' + ''.join(map(str, torch.cuda.get_device_capability(0)))
if required_arch not in torch.cuda.get_arch_list():
    raise RuntimeError(
        f'PyTorch {torch.__version__} khong co kernel {required_arch}: ' 
        f'{torch.cuda.get_arch_list()}'
    )

# Smoke test convolution de bat loi kernel ngay tai setup, truoc khi train.
test_conv = torch.nn.Conv2d(3, 4, kernel_size=3).cuda()
test_input = torch.randn(2, 3, 32, 32, device='cuda')
with torch.no_grad():
    test_conv(test_input)
torch.cuda.synchronize()
del test_conv, test_input
torch.cuda.empty_cache()

print('PyTorch:', torch.__version__)
print('Torchvision:', torchvision.__version__)
print('CUDA runtime:', torch.version.cuda)
print('CUDA architectures:', torch.cuda.get_arch_list())
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA convolution smoke test: PASSED')

In [ ]:
# Tu dong tim DATA_ROOT theo cay data/image_all, data/splits va data/processed.
# Khong hard-code slug vi Kaggle co the thay doi so cap thu muc khi mount dataset.
def find_data_root(input_root: Path = Path('/kaggle/input')) -> Path:
    candidates = []
    for image_all_dir in input_root.rglob('image_all'):
        candidate = image_all_dir.parent
        has_images = (image_all_dir / 'nih_attribute/shape').is_dir() and (image_all_dir / 'nih_attribute/color').is_dir()
        has_splits = (candidate / 'splits/nih_attribute/shape').is_dir() and (candidate / 'splits/nih_attribute/color').is_dir()
        if has_images and has_splits and (candidate / 'processed').is_dir():
            candidates.append(candidate)

    unique_candidates = sorted(set(candidates))
    if len(unique_candidates) == 1:
        return unique_candidates[0]
    if not unique_candidates:
        raise FileNotFoundError(
            'Khong tim thay DATA_ROOT. Dataset can co data/image_all/nih_attribute, '
            'data/splits/nih_attribute va data/processed trong /kaggle/input.'
        )
    raise RuntimeError(
        'Tim thay nhieu DATA_ROOT, hay chi dinh mot path: ' +
        ', '.join(str(path) for path in unique_candidates)
    )

def find_model_artifact_dir(input_root: Path = Path('/kaggle/input')) -> Path:
    # Tim dataset model attach vao Kaggle; folder can chua dung bon artifact cua last-block model.
    required = {'best.pt', 'label_mapping.json', 'optimal_thresholds.json', 'model_config.yaml'}
    candidates = sorted({path.parent for path in input_root.rglob('best.pt') if required.issubset({item.name for item in path.parent.iterdir()})})
    if len(candidates) == 1:
        return candidates[0]
    if not candidates:
        # Fallback de co the chay tren may local voi folder da dong goi trong repo.
        local_artifact_dir = REPO_DIR / 'kaggle_uploads' / 'attribute_resnet18_last_blocks_finetune'
        if required.issubset({item.name for item in local_artifact_dir.iterdir()} if local_artifact_dir.is_dir() else set()):
            return local_artifact_dir
        raise FileNotFoundError('Khong tim thay folder model co best.pt, label_mapping.json, optimal_thresholds.json va model_config.yaml.')
    raise RuntimeError('Tim thay nhieu model artifact folder. Hay chi dinh MODEL_ARTIFACT_DIR: ' + ', '.join(str(path) for path in candidates))

# Chon mot trong ba che do. Doi dong nay, khong can sua cac cell train/test ben duoi.
EXECUTION_MODE = 'test_existing_last_blocks'
# EXECUTION_MODE = 'train_head_and_last_blocks'
# EXECUTION_MODE = 'train_head_only'

if EXECUTION_MODE not in {'test_existing_last_blocks', 'train_head_and_last_blocks', 'train_head_only'}:
    raise ValueError(f'Unsupported EXECUTION_MODE: {EXECUTION_MODE}')

DATA_ROOT = find_data_root()
OUTPUT_ROOT = Path('/kaggle/working/attribute_test_runs' if EXECUTION_MODE == 'test_existing_last_blocks' else '/kaggle/working/attribute_runs')
MODEL_ARTIFACT_DIR = find_model_artifact_dir() if EXECUTION_MODE == 'test_existing_last_blocks' else None

HEAD_RUN_ID = 'attr_head_v1'
LAST_RUN_ID = 'attr_last_blocks_v1'
RUN_HEAD_TRAIN = EXECUTION_MODE in {'train_head_and_last_blocks', 'train_head_only'}
RUN_HEAD_TEST = EXECUTION_MODE in {'train_head_and_last_blocks', 'train_head_only'}
RUN_LAST_BLOCKS_TRAIN = EXECUTION_MODE == 'train_head_and_last_blocks'
RUN_LAST_BLOCKS_TEST = EXECUTION_MODE in {'test_existing_last_blocks', 'train_head_and_last_blocks'}

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO_DIR / 'src'))
print('EXECUTION_MODE:', EXECUTION_MODE)
print('DATA_ROOT:', DATA_ROOT)
print('MODEL_ARTIFACT_DIR:', MODEL_ARTIFACT_DIR)
print('OUTPUT_ROOT:', OUTPUT_ROOT)

In [ ]:
# Preflight: kiem tra schema CSV, source group leakage va synthetic chi nam o train.
from pill_safety.cv.attribute.training.data_contract import validate_attribute_data

paths = {
    'shape_image_dir': DATA_ROOT / 'image_all/nih_attribute/shape',
    'color_image_dir': DATA_ROOT / 'image_all/nih_attribute/color',
    'label_mapping': DATA_ROOT / 'processed/nih_attribute/label_mapping.json',
    'shape_train_csv': DATA_ROOT / 'splits/nih_attribute/shape/train_combined_crop.csv',
    'shape_val_csv': DATA_ROOT / 'splits/nih_attribute/shape/val_combined_crop.csv',
    'shape_test_csv': DATA_ROOT / 'splits/nih_attribute/shape/test_combined_crop.csv',
    'color_train_csv': DATA_ROOT / 'splits/nih_attribute/color/train_multilabel.csv',
    'color_val_csv': DATA_ROOT / 'splits/nih_attribute/color/val_multilabel.csv',
    'color_test_csv': DATA_ROOT / 'splits/nih_attribute/color/test_multilabel.csv',
}
manifest = validate_attribute_data(paths, verify_images=True)
manifest

In [ ]:
import json

from pill_safety.cv.attribute.training.workflow import (
    calibrate_color_thresholds,
    compare_validation_runs,
    evaluate_test,
    load_config,
    train,
)

HEAD_CONFIG = load_config(REPO_DIR / 'configs/training/attribute_resnet18_head_tune/config.yaml')
# Test-only dung dung model_config di kem checkpoint; train thi dung config train cua repo.
LAST_CONFIG = load_config(
    MODEL_ARTIFACT_DIR / 'model_config.yaml'
    if EXECUTION_MODE == 'test_existing_last_blocks'
    else REPO_DIR / 'configs/training/attribute_resnet18_last_blocks_finetune/config.yaml'
)

In [ ]:
# Phase 1 chi can khi muon train/test head. Test-only last-block bo qua toan bo pha nay.
head_checkpoint = OUTPUT_ROOT / 'attribute_resnet18_head_tune/checkpoints' / f'{HEAD_RUN_ID}_best.pt'
head_result = None
if RUN_HEAD_TRAIN:
    head_result = train(HEAD_CONFIG, str(DATA_ROOT), str(OUTPUT_ROOT), HEAD_RUN_ID)
    head_checkpoint = Path(head_result['checkpoint'])
if RUN_HEAD_TRAIN or RUN_HEAD_TEST:
    if not head_checkpoint.is_file():
        raise FileNotFoundError(f'Head checkpoint not found: {head_checkpoint}')
    head_result = {'checkpoint': str(head_checkpoint)}
    head_result
else:
    print('Head phase skipped: test-only last-block mode.')

In [ ]:
# Chi calibrate head khi head duoc train/test. Test-only last-block khong dong vao validation cua head.
head_threshold_result = None
head_thresholds = None
if RUN_HEAD_TRAIN or RUN_HEAD_TEST:
    head_threshold_result = calibrate_color_thresholds(HEAD_CONFIG, head_checkpoint, str(DATA_ROOT), str(OUTPUT_ROOT), HEAD_RUN_ID)
    head_thresholds = Path(head_threshold_result['path'])
    head_threshold_result
else:
    print('Head calibration skipped: test-only last-block mode.')

In [ ]:
# Test chi la reporting sau khi checkpoint va threshold da duoc chon bang validation.
# Test khong duoc dung de tune threshold, chon epoch hay chon chien luoc fine-tune.
if RUN_HEAD_TEST:
    assert head_threshold_result['split'] == 'validation'
    head_test = evaluate_test(HEAD_CONFIG, head_checkpoint, head_thresholds, str(DATA_ROOT), str(OUTPUT_ROOT), HEAD_RUN_ID)
    assert head_test['split'] == 'test'
    print('=== HEAD-TUNE: TEST METRICS ===')
    print(json.dumps(head_test['metrics'], indent=2))
    print('Per-class shape F1:', json.dumps(head_test['per_class_metrics']['shape'], indent=2))
    print('Per-color F1:', json.dumps(head_test['per_class_metrics']['color'], indent=2))
    print('Test metric file:', head_test['path'])
    print('Test plots:', Path(head_test['path']).parent.parent / 'plots')
    print('Prediction examples:', Path(head_test['path']).parent.parent / 'predictions' / HEAD_RUN_ID)
else:
    print('Head test skipped: RUN_HEAD_TEST=False')

In [ ]:
# Phase 2: test-only nap thang best.pt da dong goi; neu train thi nap head best va fine-tune layer3/layer4.
last_checkpoint = None
last_result = None
if RUN_LAST_BLOCKS_TRAIN or RUN_LAST_BLOCKS_TEST:
    last_checkpoint = (
        MODEL_ARTIFACT_DIR / 'best.pt'
        if EXECUTION_MODE == 'test_existing_last_blocks'
        else OUTPUT_ROOT / 'attribute_resnet18_last_blocks_finetune/checkpoints' / f'{LAST_RUN_ID}_best.pt'
    )
    if RUN_LAST_BLOCKS_TRAIN:
        if not head_checkpoint.is_file():
            raise FileNotFoundError(f'Head checkpoint required for last-block training: {head_checkpoint}')
        last_result = train(LAST_CONFIG, str(DATA_ROOT), str(OUTPUT_ROOT), LAST_RUN_ID, pretrained_override=str(head_checkpoint))
        last_checkpoint = Path(last_result['checkpoint'])
    if not last_checkpoint.is_file():
        raise FileNotFoundError(f'Last-block checkpoint not found: {last_checkpoint}')

    # Test-only phai kiem tra model va test data dung cung label order.
    if EXECUTION_MODE == 'test_existing_last_blocks':
        artifact_mapping = json.loads((MODEL_ARTIFACT_DIR / 'label_mapping.json').read_text(encoding='utf-8'))
        data_mapping = json.loads((DATA_ROOT / 'processed/nih_attribute/label_mapping.json').read_text(encoding='utf-8'))
        if artifact_mapping != data_mapping:
            raise ValueError('label_mapping cua model artifact khong khop label_mapping cua test dataset.')
    last_result = {'checkpoint': str(last_checkpoint), 'mode': 'train' if RUN_LAST_BLOCKS_TRAIN else 'test_only'}
    last_result
else:
    print('Last-block phase skipped: head-only training mode.')

In [ ]:
# Test-only dung nguyen threshold da calibrate tren validation va dong goi cung model.
# Khong chay lai calibration tren validation de giu nguyen model artifact duoc danh gia.
if RUN_LAST_BLOCKS_TRAIN:
    last_threshold_result = calibrate_color_thresholds(LAST_CONFIG, last_checkpoint, str(DATA_ROOT), str(OUTPUT_ROOT), LAST_RUN_ID)
    last_thresholds = Path(last_threshold_result['path'])
elif RUN_LAST_BLOCKS_TEST:
    last_thresholds = MODEL_ARTIFACT_DIR / 'optimal_thresholds.json'
    if not last_thresholds.is_file():
        raise FileNotFoundError(f'Optimal thresholds not found: {last_thresholds}')
    last_threshold_result = json.loads(last_thresholds.read_text(encoding='utf-8'))
else:
    last_thresholds = None
    last_threshold_result = None
    print('Last-block calibration/test skipped: head-only training mode.')

if RUN_LAST_BLOCKS_TEST:
    assert last_threshold_result['split'] == 'validation'
    last_test = evaluate_test(LAST_CONFIG, last_checkpoint, last_thresholds, str(DATA_ROOT), str(OUTPUT_ROOT), LAST_RUN_ID)
    assert last_test['split'] == 'test'
    print('=== LAST-BLOCKS: TEST METRICS ===')
    print(json.dumps(last_test['metrics'], indent=2))
    print('Per-class shape F1:', json.dumps(last_test['per_class_metrics']['shape'], indent=2))
    print('Per-color F1:', json.dumps(last_test['per_class_metrics']['color'], indent=2))
    print('Test metric file:', last_test['path'])
    print('Test plots:', Path(last_test['path']).parent.parent / 'plots')
    print('Prediction examples:', Path(last_test['path']).parent.parent / 'predictions' / LAST_RUN_ID)
else:
    print('Last-block test skipped: RUN_LAST_BLOCKS_TEST=False')

In [ ]:
# Chi so sanh/selection khi notebook vua train ca head va last-block. Test-only khong phat sinh selection moi.
if RUN_HEAD_TRAIN and RUN_LAST_BLOCKS_TRAIN:
    comparison = compare_validation_runs(
        OUTPUT_ROOT / 'attribute_resnet18_head_tune/metrics' / f'{HEAD_RUN_ID}_val_metrics.json',
        OUTPUT_ROOT / 'attribute_resnet18_last_blocks_finetune/metrics' / f'{LAST_RUN_ID}_val_metrics.json',
        OUTPUT_ROOT / 'attribute_model_selection.json',
    )
    comparison
else:
    print('Validation model selection skipped: test-only uses the supplied last-block checkpoint.')